In [0]:
base_path = "/Volumes/workspace/default/uc_fund_data_2025_bronze"
folders = dbutils.fs.ls(base_path)
for folder in folders:
  print(f"folder {folder.name} has {dbutils.fs.ls(folder.path)} files")


Testing the file conversion into tables

In [0]:
base_path = "/Volumes/workspace/default/uc_fund_data_2025_bronze/"
fund_folders = dbutils.fs.ls(base_path)

# Create bronze schema
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze_fund_data")


for folder in fund_folders:
    files = dbutils.fs.ls(folder.path)
    folder_name = folder.name.replace("/", "").lower()
    

    for file in files:
        if file.name.endswith('.txt'):
            file_name = file.name.replace('.txt', '').lower()
            
            try:
                # Detect delimiter
                first_line = spark.read.text(file.path).first().value
                delimiter = "|" if first_line.count('|') > first_line.count('\t') else "\t"
                
                # Read file
                df = spark.read \
                    .option("header", "true") \
                    .option("delimiter", delimiter) \
                    .csv(file.path)
                
                # Add metadata columns for lineage
                from pyspark.sql.functions import lit
                df = df.withColumn("fund_id", lit(folder_name)) \
                       .withColumn("source_file", lit(file.name)) 
                
                #create table name - folder_name_file_name
                table_name = f"workspace.bronze_fund_data.{folder_name}_{file_name}"
                df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)
            
                print(f" Fixed {table_name}: {df.count()} rows, {len(df.columns)} columns")

            
            except Exception as e:
                print(f"  FAILED {folder_name}/{file.name}: {e}")
                continue

print("\n=== Bronze layer fixed ===")

        


In [0]:
import pandas as pd

output_path = "/Volumes/workspace/default/uc_fund_data_2025_bronze"
rows = []
for folder, files in fund_folders_file_shapes.items():
    for file, stats in files.items():
        rows.append({
            'Folder': folder,
            'File': file,
            'Row Count': stats.get('row_cnt', 0),
            'Column Count': stats.get('col_cnt', 0)
        })

df = pd.DataFrame(rows)
display(df)
df.to_csv(f'{output_path}/fund_shapes.csv', index=False)

print("✓ Done! Download from Databricks: Data → DBFS → FileStore → fund_shapes.xlsx")

In [0]:
folder_120A = '/Volumes/workspace/default/uc_fund_data_2025_bronze/120A/'
file_info = dbutils.fs.ls(folder_120A)

dfs_12A = {}

for file in file_info:
  file_path = file.path
  file_name = file.name.replace(".txt","")
  try:
    df = spark.read.option("header", "true").option("delimiter", "|").option("inferSchema","true").csv(file_path)
    dfs_12A[file_name] = df
    print(f"Read file {file_path} into dataframe {file_name}")

    if len(df.columns) == 1:
      print(f"File {file_name} has 1 column.")

      df = spark.read.option("header", "true").option("delimiter", "\t").option("inferSchema", "true").csv(file_path)

    dfs_12A[file_name] = df
  except Exception as e:
    print(f"Error reading file {file_path}: {e}")
  

In [0]:
for df in dfs_12A.keys():
  print(f"df {df} headers \n {dfs_12A[df].show(3)} \n and the columns are {dfs_12A[df].columns}")


In [0]:
%sql
ALTER TABLE workspace.bronze_fund_data.program_code_desc
SET LOCATION "dbfs:/Volumes/workspace/default/uc_fund_data_2025_bronze/Code_Details/";